In [0]:
%run ./main

In [0]:
# IMPORTS

from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql.window import Window

In [0]:
# Null check

orders_df.select([
    F.count(F.when(
        F.col(c).isNull(), c
    )).alias(c) for c in orders_df.columns
]).show()

In [0]:
for c in orders_df.columns:
    print(c)

In [0]:
# duplicate check if order_id is duplicate
dup_order = orders_df.groupBy("order_id").count()
dup_order.show()

dup_order = dup_order.filter(F.col("count") > 1)
dup_order.show()

In [0]:
# joining multiple tables

order_details = orders_df.join(
    products_dataframe, "product_id"
).join(
    customer_dataframe, "customer_id"
)

order_details.select(
    "order_id", 
    "title", 
    col("name").alias("customer_name"), 
    "price", 
    orders_df.status.alias("order_status")
).show()


display(order_details)
                        

In [0]:
# top 3 products by revenue

"""
first we will calculate the revenue in order by grouping with product id and title
then we will order them in desc and then show last 3

NOTE: order_details is joined with product and customers
"""

top_products_by_revenue = order_details.withColumn(
        "revenue", F.col("price") * F.col("quantity")
    ).groupBy("product_id", "title").agg(
        F.sum("revenue").alias("total_revenue")
    ).orderBy(
        F.desc("total_revenue")
    )
top_products_by_revenue.show(3)

In [0]:
# Top 2 Products by Revenue Per Country
country_revenue = order_details.withColumn(
        "revenue", F.col("price") * F.col("quantity")
    ).groupBy("country", "product_id", "title").agg(
        F.sum("revenue").alias("total_revenue")
    )
country_revenue.show()

w = Window.partitionBy("country").orderBy(F.desc("total_revenue"))
top2_country = country_revenue.withColumn("rank", F.row_number().over(w)).filter(F.col("rank") <= 2)
top2_country.show()

top2_w = Window.orderBy(F.desc("total_revenue"))
top2_overall = country_revenue.withColumn("rank", F.row_number().over(top2_w)).filter(
    F.col("rank") <= 2
)
top2_overall.show()

In [0]:
# Explode — Handling Nested Items

nested_orders = spark.createDataFrame([
    (2001, 1, [
        {"product_id": "101", "qty": "1"},
        {"product_id": "103", "qty": "2"}
    ])
], ["order_id", "customer_id", "items"])
nested_orders.show()

exploded = nested_orders.select(
        "order_id", "customer_id", 
        F.explode("items").alias("item")
    ).select(
        "order_id", "customer_id",
        F.col("item.product_id").cast("int").alias("product_id"),
        F.col("item.qty").cast("int").alias("quantity")
    )

exploded.show()

In [0]:
# Handling Array Columns (Product Tags Example)

product_tags = [
    (101, ["smart", "speaker", "voice"]),
    (102, ["ebook", "reader"]),
    (103, ["home", "kitchen"]),
    (104, ["fitness", "exercise"]),
]

tag_schema = ["product_id", "tags"]

tags_df = spark.createDataFrame(product_tags, tag_schema)

tags_df.select("product_id", F.explode("tags").alias("tag")).show()

In [0]:
# Group Aggregations with Multiple Metrics

# count, total quantity and average quantity on the basis of status

order_agg = orders_df.groupBy("status").agg(
    F.count("*").alias("total_count"),
    F.sum("quantity").alias("total_quantity"),
    F.avg("quantity").alias("avg_qty")
)
order_agg.show()

In [0]:
# PIVOT - Why: Pivoting turns rows into columns — great for reporting.
# Pivot (Status by Country)
pivot_df = order_details.groupBy("country").pivot("status").count().fillna(0)
pivot_df.show()

In [0]:
# Join with Composite Keys

inventory = [
    (101, "us-east-1", 500),
    (101, "eu-west-1", 200),
    (102, "us-east-1", 150),
    (103, "us-east-1", 1000),
    (104, "ap-south-1", 300),
]
inventory_df = spark.createDataFrame(inventory, ["product_id", "warehouse", "qty"])

mapping = spark.createDataFrame([(101, "us-east-1"), (102, "us-east-1")], ["product_id", "warehouse"])

inventory_df.join(mapping, ["product_id", "warehouse"], "inner").show()

In [0]:
# Time-Series — Daily Orders

# Why: Date grouping helps you track daily activity.

orders_df.select(F.to_date("order_date").alias("date")).groupBy("date").count().orderBy("date").show()

In [0]:
data = [(f"2023-07-{day:02d}", int(day * 10)) for day in range(1, 6)]
daily_df = spark.createDataFrame(data, ["date", "orders"])

w = Window.orderBy("date").rowsBetween(-2, 0)
daily_df.withColumn("rolling_avg", F.avg("orders").over(w)).show()

In [0]:
# Using spark in sql

orders_df.createOrReplaceTempView("orders")
products_dataframe.createOrReplaceTempView("products")

spark.sql("""
SELECT p.title, sum(o.quantity * p.price) AS revenue
FROM orders o
JOIN products p USING (product_id)
GROUP BY p.title
ORDER BY revenue DESC
""").show()

In [0]:
# Session Analytics (Simple Example)
sessions = [
    ("s1", 1, "2023-07-01 10:00:00", "view", 101),
    ("s2", 2, "2023-07-02 12:00:00", "search", None),
    ("s3", 1, "2023-07-03 09:00:00", "purchase", 101),
    ("s4", 1, "2023-07-03 09:00:00", "timepass", None),
]
sessions_df = spark.createDataFrame(sessions, ["session_id", "customer_id", "event_time", "event_type", "product_id"])

# Count events per customer
sessions_df.groupBy("customer_id").pivot("event_type").count().fillna(0).show()

In [0]:
# Delta Table Basics (Databricks)


orders_df.write.format("delta").mode("overwrite").save("/Volumes/pyspark_cat/ecommerce/ecom_volume/orders_delta")
orders_delta = spark.read.format("delta").load("/Volumes/pyspark_cat/ecommerce/ecom_volume/orders_delta")
orders_delta.show()


In [0]:
orders_delta.select("product_id", col("quantity").alias("order_qty")).show()

In [0]:
from delta.tables import DeltaTable

deltaTable = DeltaTable.forPath(spark, "/Volumes/pyspark_cat/ecommerce/ecom_volume/orders_delta")

updates_in_df = spark.createDataFrame([
    (1005, 4, 103, 1, "2023-08-01", "delivered"),   # new order
    (1004, 3, 104, 3, "2023-08-01", "returned")     # update order
], orders_schema)

deltaTable.alias("t").merge(
    updates_in_df.alias("s"),
    "t.order_id = s.order_id"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
orders_delta.orderBy("order_id").show()